In [ ]:
##Author Liu Xiaoquan Ace, Asistant Professor, HKCHC
#Pro-worker智能体代码结构示意

In [ ]:
pip install zhipuai pandas faiss-cpu numpy

In [ ]:
import os
from zhipuai import ZhipuAI

# 设置您的 API Key
# 建议通过环境变量管理密钥
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY", "YOUR_API_KEY_HERE")

class GLMConnector:
    def __init__(self, model="glm-*"): # 或根据实际可用模型选择 glm-5 系列
        self.client = ZhipuAI(api_key=ZHIPU_API_KEY)
        self.model = model

    def generate(self, system_prompt: str, user_prompt: str, temperature=0.7):
        """
        核心生成函数
        """
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature
        )
        return response.choices[0].message.content

# 初始化连接器
llm = GLMConnector()

In [ ]:
class DynamicRAG:
    def __init__(self):
        # 模拟 2025-2026 年的行业知识库
        self.knowledge_base = [
            "2025年趋势：生成式AI从'替代'转向'副驾驶'模式，需求侧重点关注AI训练师、提示词工程师。",
            "2026年预测：随着自动化程度加深，'机器共情能力'与'复杂决策能力'成为人类核心价值点。",
            "Lightcast报告：技术变革导致传统文员岗位下降，但AI维护与人机协作岗位缺口增加40%。"
        ]

    def retrieve(self, query: str, k=2):
        """
        模拟检索过程：根据查询返回最相关的上下文
        """
        # 在实际应用中，此处应使用 faiss 或 elasticsearch 进行向量检索
        relevant_context = [doc for doc in self.knowledge_base if any(word in doc for word in query.split())]
        return "\n".join(relevant_context[:k])

rag_system = DynamicRAG()

In [ ]:
from abc import ABC, abstractmethod

class BaseAgent(ABC):
    def __init__(self, name, llm_connector, rag_system):
        self.name = name
        self.llm = llm_connector
        self.rag = rag_system

    @abstractmethod
    def execute(self, input_data):
        pass

In [ ]:
class MarketAnalystAgent(BaseAgent):
    def execute(self, industry_query: str):
        print(f"--- {self.name} 正在分析市场数据 ---")
        
        # 1. 通过 RAG 获取最新数据
        context = self.rag.retrieve(industry_query)
        
        # 2. 构造 Prompt
        system_prompt = "你是一个宏观劳动力市场分析师。基于提供的市场数据，识别新兴的技能缺口和岗位机会。"
        user_prompt = f"""
        当前市场数据上下文：
        {context}
        
        用户查询：{industry_query}
        
        请列出该领域最新出现的3个关键技能缺口。
        """
        
        return self.llm.generate(system_prompt, user_prompt)

In [ ]:
class StrategicHRAgent(BaseAgent):
    def execute(self, skill_gaps: str):
        print(f"--- {self.name} 正在设计岗位 ---")
        
        system_prompt = "你是企业战略HR专家。请根据技能缺口设计具体的岗位描述，包括职责和所需技能。"
        user_prompt = f"""
        识别到的技能缺口：{skill_gaps}
        
        请设计一个能够填补上述缺口的岗位。
        """
        
        return self.llm.generate(system_prompt, user_prompt)

In [ ]:
class ProWorkerEvaluatorAgent(BaseAgent):
    def execute(self, job_description: str):
        print(f"--- {self.name} 正在进行 Pro-Worker 审计 ---")
        
        # 核心理论注入：Acemoglu Utility Curve 算法化
        system_prompt = """
        你是一个遵循 Acemoglu (2025) "Pro-Worker" 原则的审计员。
        你的目标是最大化劳动力效用。
        
        评估标准：
        1. 检查岗位是否属于 "Excessive Automation" (过度自动化/单纯替代人类)。
        2. 若岗位旨在替代人类，驳回并要求修改为 "Augmentation Role" (增强型岗位)。
        3. 确保岗位包含 "New Tasks" (新任务)，即利用 AI 增强人类技能的维度。
        
        输出格式：
        - Status: [APPROVED / REJECTED]
        - Feedback: [修改建议或通过理由]
        - Augmentation_Score: [0-10, 10代表完全增强人类能力]
        """
        
        user_prompt = f"待审计岗位描述：\n{job_description}"
        
        return self.llm.generate(system_prompt, user_prompt, temperature=0.3) # 降低温度以保证逻辑严谨性

In [ ]:
class BenchmarkingAgent(BaseAgent):
    def execute(self, final_job_desc: str, previous_results: list):
        # 简化的 Benchmark 逻辑
        print(f"--- {self.name} 正在进行基准测试 ---")
        system_prompt = "你是一个质量审计员。对比生成的岗位与行业标准，评估其逻辑一致性。"
        # 实际代码中可对比不同模型输出
        return f"Benchmark Report: Job definition is logically consistent with 2025 trends. \n{final_job_desc}"

In [ ]:
def run_pro_worker_framework(query: str):
    # 初始化所有智能体
    maa = MarketAnalystAgent("MAA", llm, rag_system)
    sha = StrategicHRAgent("SHA", llm, rag_system)
    pwea = ProWorkerEvaluatorAgent("PWEA", llm, rag_system)
    ba = BenchmarkingAgent("BA", llm, rag_system)
    
    # 1. 市场分析
    gaps = maa.execute(query)
    
    # 2. 岗位设计
    job_desc = sha.execute(gaps)
    
    # 3. Pro-Worker 评估循环
    # 模拟 Acemoglu 的迭代优化过程
    max_iterations = 3
    for i in range(max_iterations):
        print(f"\nIteration {i+1}...")
        evaluation = pwea.execute(job_desc)
        
        if "APPROVED" in evaluation:
            print(">> 岗位已通过 Pro-Worker 审计。")
            break
        else:
            print(">> 岗位被驳回，原因：过度自动化。正在重新设计...")
            # 将审计反馈作为新输入传回 SHA
            job_desc = sha.execute(f"基于之前的驳回意见重新设计：{evaluation}")
    else:
        print(">> 达到最大迭代次数，采用当前最优版本。")
        
    # 4. 基准测试
    final_report = ba.execute(job_desc, [])
    
    return final_report

# --- 执行示例 ---
if __name__ == "__main__":
    result = run_pro_worker_framework("2025年生成式AI在金融领域的应用趋势")
    print("\n======== 最终结果 ========")
    print(result)